# 04.02 - VLM specifics: vision encoder + language model

**Why.** A VLM is not just an LLM. It carries a **vision encoder** in front of the language model,
and that extra part changes what the compiler produces and what the runtime checks. This notebook is
the VLM-specific companion to `01_llm_compilation.ipynb`; read that one first for the shared honesty
boundary (there is no `llima compile`; compilation is the host-side `llima-compile` Model Compiler
step) and the shared source-format / quantization rules.

Concept + documented-procedure only. Heavy commands are printed strings; nothing here runs a model.

## What makes a VLM different: two subgraphs

A vision-language model has two cooperating parts:

1. **Vision encoder** - turns an image into embeddings (the `VISION` part in the compiler's
   part-list).
2. **Language model** - the usual transformer decoder that consumes text tokens *and* the image
   embeddings and generates text.

The compiler splits a model into parts; the docs expose a `part` field with values
`{PRE, CACHE, POST, VISION}`. The `VISION` part is the encoder; `PRE`/`CACHE`/`POST` are the
language-model stages. For a text-only LLM there is simply **no `VISION` part**. That single
structural difference is what the runtime keys on to decide "text-only vs image-capable".

## Supported VLM architectures

| Family | Examples in catalog |
| --- | --- |
| LLaVA 1.5 | `llava-1.5-7b-hf-a16w4` |
| PaliGemma | `paligemma-3b-pt-224-a16w8` |
| Gemma 3 / 4 (multimodal) | `gemma3-siglip448-a16w4` |
| Qwen2.5-VL / Qwen3-VL | `Qwen3-VL-4B-Instruct-GPTQ-a16w4` |
| LFM2-VL / LFM2.5-VL | `LFM2-VL-1.6B-a16w4` |

Same rule as LLMs: the *family* must be supported. The board already has
`Qwen3-VL-4B-Instruct-GPTQ-a16w4` pulled - use it for any VLM happy path; no pull needed.

## Source-format requirements for VLMs

The three accepted formats are the same as for LLMs (HF safetensors, GGUF, GPTQ/AWQ), but a
VLM source must additionally carry its **vision configuration**:

- The HF `config.json` must describe the vision tower (e.g. a `vision_config` / SigLIP / CLIP block),
  and the processor/image-preprocessing config must be present so the encoder's expected input size
  and normalization are known.
- The four companion files from `01_llm_compilation.ipynb` (`config.json`, `tokenizer.json`,
  `tokenizer_config.json`, `generation_config.json`) are still required.
- Pre-quantized VLMs (e.g. the Qwen-VL GPTQ variants) follow the same GPTQ/AWQ rules as LLMs (confirm any symmetry requirement against the docs).

If the vision half is missing or mis-declared, the compiler cannot build a `VISION` part and you end
up with a text-only model - see failure modes below.

### The compile command (documented, NOT executed)

Identical tool to the LLM case - `llima-compile` decides LLM vs VLM from the source's config. Printed
string only; this step runs host-side on the Model Compiler, not on the board.

In [ ]:
# Documented shape only, from
# https://developer.sima.ai/software/genai-llima/compilation_genai
# llima-compile is a host-side Model Compiler tool; confirm flags against the docs.

vlm_compile_cmd = "llima-compile [options] <path-to-vlm-source>"

# For a VLM, expect the output to include a VISION part in addition to the
# language-model parts (PRE / CACHE / POST). Precision knob and 1024-token
# default context are the same as the LLM path.
print("Documented command shape (runs on a host with the Model Compiler, not the board):")
print(" ", vlm_compile_cmd)

## The runtime contract for a VLM

This is the part we can state with certainty, because it is enforced in Neat core source
([`GenAIInternal.cpp`](https://github.com/sima-neat/core/blob/main/src/genai/GenAIInternal.cpp), `inspect_model_directory` +
`has_vision_capability`). A deployed VLM directory must have:

```text
/media/nvme/llima/models/<vlm-id>/
  devkit/
    vlm_config.json     # NOT whisper_config.json
  elf_files/            # compiled binaries incl. the vision encoder
```

And crucially, for the model to be **image-capable**, `devkit/vlm_config.json` must contain:

- `vm_cfg`  - non-null
- `mm_cfg`  - non-null
- `vision_model_name` - a non-empty string

The runtime function `has_vision_capability()` returns `True` only when **all three** hold; that is
exactly what sets `accepts_image()`. A `vlm_config.json` without them loads as a **text-only LLM**.

In [ ]:
# The exact capability rule, transcribed from
# https://github.com/sima-neat/core/blob/main/src/genai/GenAIInternal.cpp (has_vision_capability).
# This is documentation-as-code; it constructs no model and is safe to run.

def is_image_capable(vlm_config: dict) -> bool:
    return (
        vlm_config.get("vm_cfg") is not None
        and vlm_config.get("mm_cfg") is not None
        and isinstance(vlm_config.get("vision_model_name"), str)
        and vlm_config.get("vision_model_name") != ""
    )

# Illustrative only (fabricated minimal dicts, not real config contents):
print("with vision keys ->", is_image_capable(
    {"vm_cfg": {}, "mm_cfg": {}, "vision_model_name": "siglip"}))
print("missing vision   ->", is_image_capable(
    {"vm_cfg": None, "mm_cfg": None, "vision_model_name": ""}))

## Common failure modes

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `GenAIModel` loads but `accepts_image()` is `False` | `vlm_config.json` lacks `vm_cfg` / `mm_cfg` / non-empty `vision_model_name` - the vision half did not compile in | re-check the source `config.json` vision block; recompile so the `VISION` part is produced |
| Runtime throws `"missing elf_files/"` | deploy copied `devkit/` but not the compiled binaries | deploy the full tree; `elf_files/` must be present |
| Runtime throws `"has both vlm_config.json and whisper_config.json"` | a stray whisper config in `devkit/` | a model dir is LLM/VLM **or** ASR, never both - remove the wrong config |
| Image output is garbage / wrong colors | image passed as BGR | VLM images must be **uint8 HWC RGB**; `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` before adding (OpenCV decodes BGR) |
| Compile OOM on the host during quantize | vision + LM together is large | GPU recommended; drop to `a16w4`; pick a smaller VLM variant |
| Board runs out of disk on deploy | VLM weights bigger than an LLM of the same param count | `df -h /media/nvme` first; prefer `LFM2-VL-450M-a16w4` for a small fresh VLM |

## Expected artifacts

After a successful VLM compile + deploy:

- Compiler output includes a `VISION` part alongside the language-model parts.
- Deployed `/media/nvme/llima/models/<vlm-id>/` has `devkit/vlm_config.json` (with non-null `vm_cfg`,
  `mm_cfg`, and a non-empty `vision_model_name`) and `elf_files/`.
- Smoke test (see `02-run-llm-vlm/02_run_vlm.ipynb`):
  - `m = pyneat.genai.VisionLanguageModel("/media/nvme/llima/models/<vlm-id>")`
  - `m.accepts_image()` -> `True`
  - a text+image request returns a description of the image.

If `accepts_image()` is `False`, the vision encoder did not make it into the model - treat it as a
compile problem, not a runtime one.